In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
sonawanelalitsunil_global_electric_vehicle_trends_path = kagglehub.dataset_download('sonawanelalitsunil/global-electric-vehicle-trends')

print('Data source import complete.')


In [ ]:
!pip install --upgrade pandas

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option('display.float_format', lambda x: '%.f' % x)
pd.set_option('display.max_rows', None)     # Display all rows
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', None)        # Allow the display width to be unlimited (often used with max_columns)
pd.set_option('display.max_colwidth', None) # Display full content of each cell (no truncation)

# Read Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/global-electric-vehicle-trends/Electric_Vehicle_Population_Data (1).csv')
data = df.copy()
df.head()

# Data Preprocessing

In [ ]:
df.info()

In [ ]:
# Removing unwanted columns
df = df.drop(columns = ['Postal Code','Legislative District','DOL Vehicle ID','VIN (1-10)', '2020 Census Tract', 'Electric Utility'])

# Finding missing values in  'County'  and 'City' columns

In [ ]:
# Identifying null values in "County"
null_count = df['County'].isnull().sum()
print(null_count)

# Before deleting these 10 rows, Checking if there is a possibility to replace 'county'
# with max count 'county' in its corresponding 'City/state'.

In [ ]:
# Finding if both 'County' and 'City' columns are null
df[
    (df['City'].isnull()) &  # Condition 1: Value equals the target
    (df['County'].isnull())
]

In [ ]:
# Calculating Value counts for each unique value in 'State'
each_state_counts = df.groupby('State')['County'].value_counts()
# print(each_state_counts)

In [ ]:
state_unique_values = set(df['State'].dropna().unique())
state_values_with_county_data  =  set(each_state_counts.index.get_level_values(0).unique())
missing_primary_values = state_unique_values - state_values_with_county_data
print(missing_primary_values)

In [ ]:
# Deleting the rows with NaN values
rows_to_keep = ~df['State'].isin(missing_primary_values)
df_cleaned = df[rows_to_keep].copy()
print(f"Original number of rows: {len(df)}")
print(f"Number of rows deleted: {len(df) - len(df_cleaned)}")
print(f"New number of rows: {len(df_cleaned)}")

In [ ]:
df['County'].nunique()
# df['County'].value_counts()

In [ ]:
number_of_singletons = (df['County'].value_counts() == 1).sum()
print(number_of_singletons)

# Seperating Latitude and Longitude Values

In [ ]:
df_cleaned[['Point', 'Latitude', 'Longitude']] = (
    df_cleaned['Vehicle Location']
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.split(' ', expand=True)
)
df_cleaned = df_cleaned.drop(columns = ['Point','Vehicle Location'])
df_cleaned.head()

# Simplifying data in Clean Alternative Fuel Vehicle (CAFV) Eligibility column

In [ ]:
df_cleaned['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()

In [ ]:
df_cleaned['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] = (
    df_cleaned['Clean Alternative Fuel Vehicle (CAFV) Eligibility']
    .str.replace('Eligibility unknown as battery range has not been researched', 'Unknown', regex = True)
    .str.replace('Clean Alternative Fuel Vehicle Eligible','Eligible', regex = True)
    .str.replace('Not eligible due to low battery range','Not Eligible', regex = True)
)

# Neglecting Base MSRP column as most of the values are zeros

In [ ]:
# Majority of values are Zeros
# Neglecting the column
df_cleaned['Base MSRP'].value_counts()

In [ ]:
Zero_MSRP_values = df_cleaned[df_cleaned['Base MSRP'] == 0][['Make', 'Model']].value_counts().sort_index()
Non_Zero_MSRP_values = df_cleaned[df_cleaned['Base MSRP'] != 0][['Make', 'Model']].value_counts().sort_index()
df_cleaned = df_cleaned.drop(columns = 'Base MSRP')

# Data Visualisation

# 1. Annual Growth of Electric Vehicle Sales

In [ ]:
df_cleaned['Model Year'].value_counts().reset_index().head(15)
# Considering data from 2011 for plotting

In [ ]:
# Create the bar plot
year_counts = df_cleaned['Model Year'].value_counts().reset_index().head(15)
year_counts.columns = ['Model Year', 'Count']

# Plot the bar graph
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

bar_plot = sns.barplot(
    x='Model Year',
    y='Count',
    data=year_counts,
    palette='Blues_d'
)

# Add title and labels
plt.title('Electric Vehicle Sales', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=14)
plt.ylabel('Number of Electric Vehicles', fontsize=14)

plt.tight_layout()


# 2. EV Sales Distribution Across Major Brands

In [ ]:
df_cleaned['Make'].value_counts().reset_index().head(20)

In [ ]:
# Create the bar plot
# Considering top 15 Brands in sales for plotting
brand_counts = df_cleaned['Make'].value_counts().reset_index().head(15)
brand_counts.columns = ['Make', 'Count']

# Plot the bar graph
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

bar_plot = sns.barplot(
    x='Make',
    y='Count',
    data=brand_counts,
    palette='Blues_d'
)

# Add title and labels
plt.title('EV Sales Distribution Across Major Brands', fontsize=16, fontweight='bold')
plt.xlabel('Brand', fontsize=14)
plt.ylabel('Number of Electric Vehicles', fontsize=14)
plt.xticks(rotation=45, ha='center')
plt.tight_layout()

# 3. EV Brand Performance Analysis: (2015 - 2025)

In [ ]:
brand_counts_by_year = df_cleaned.groupby(['Make', 'Model Year']).size().reset_index(name='Count').sort_values(by='Make')

brand_year_count = brand_counts_by_year.groupby('Make')['Model Year'].nunique().reset_index()
brand_year_count.columns = ['Make', 'Unique Year Count']

popular_brands = brand_year_count[brand_year_count['Unique Year Count'] > 10]
print(popular_brands)
popular_makes = popular_brands['Make'].tolist()

# Filter the data to include only the popular brands
df_plot = brand_counts_by_year[brand_counts_by_year['Make'].isin(popular_makes)]
df_recent = df_plot[(df_plot['Model Year']>=2015) & (df_plot['Model Year']<=2025)]
df_pivot = df_recent.pivot(index='Model Year', columns='Make', values='Count').fillna(0)

# Plot the line graph
plt.figure(figsize=(12, 6))
df_pivot.plot(kind='line',
              ax=plt.gca(), # Use the current Axes object
              linewidth=2,
              marker='o',
              markersize=4)

# Customization
plt.title('EV Brand Performance Analysis (2015-2025)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=14)
plt.ylabel('Annual Count (Units)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Set the x-axis ticks to show only integer years
plt.xticks(df_pivot.index.values)

# Move the legend outside the plot area for cleaner visualization
plt.legend(title='Car Make', loc='upper left', bbox_to_anchor=(1.01, 1.0), fancybox=True, shadow=True)

plt.tight_layout(rect=[0, 0, 0.95, 1]) # Adjust layout to ensure legend fits
plt.show()


# 4. BEV and PHEV: A Comparative Study on EV

In [ ]:
df_cleaned['Electric Vehicle Type'].value_counts()

In [ ]:
# Type of Electric Vehicles Produce by different brands
Vehicle_type_by_brands=df_cleaned.groupby(['Make', 'Electric Vehicle Type']).size().reset_index(name='Count').sort_values(by='Make')

df_type = Vehicle_type_by_brands[Vehicle_type_by_brands['Make'].isin(popular_makes)]
df_pivot_type = df_type.pivot(index='Make', columns='Electric Vehicle Type', values='Count').fillna(0)
plt.figure(figsize=(14, 7))

# Plot directly from the pivoted DataFrame.
# kind='bar' creates bar charts.
# stacked=True ensures the bars are stacked on top of each other.
df_pivot_type.plot(kind='bar',
                   stacked=True,
                   ax=plt.gca(),
                   edgecolor='black')

# Customization
plt.title('Global EV Sales Performance: Manufacturer vs. Vehicle Type', fontsize=16, fontweight='bold')
plt.xlabel('Brands', fontsize=14)
plt.ylabel('Total Vehicle Count (Units)', fontsize=14)
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

# 5. Brand-Specific Analysis of Clean Alternative Fuel Vehicle Status

In [ ]:
df_cleaned['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()

In [ ]:
CAFV_eligibility=df_cleaned.groupby(['Make', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility']).size().reset_index(name='Count').sort_values(by='Make')
df_CAFV = CAFV_eligibility[CAFV_eligibility['Make'].isin(popular_makes)]
df_pivot_CAFV = df_CAFV.pivot(index='Make', columns='Clean Alternative Fuel Vehicle (CAFV) Eligibility', values='Count').fillna(0)
plt.figure(figsize=(14, 8))

# Plot directly from the pivoted DataFrame.
# kind='bar' creates bar charts.
# stacked=True ensures the bars are stacked on top of each other.
df_pivot_CAFV.plot(kind='bar',
                   stacked=True,
                   ax=plt.gca(),
                   edgecolor='black')

# Customization
plt.title('Comparative Study of EV Brands and CAFV Compliance', fontsize=16, fontweight='bold')
plt.xlabel('Brands', fontsize=14)
plt.ylabel('Total Vehicle Count (Units)', fontsize=14)
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

# 6. Assessing Electric Range disparity among popular EV brands



In [ ]:
ERange=df_cleaned.groupby(['Make', 'Electric Range' ]).size().reset_index(name='Count').sort_values(by='Make')
df_ERange = ERange[ERange['Make'].isin(popular_makes)]
df_ERange = df_ERange[df_ERange['Electric Range'] > 0]

make_categories = df_ERange['Make'].astype('category')
make_indices = make_categories.cat.codes
make_names = make_categories.cat.categories.tolist()

np.random.seed(1)
jitter = np.random.uniform(-0.15, 0.15, size=len(df_ERange))
x_plot = make_indices + jitter
plt.figure(figsize=(12, 7))

# Create the scatter plot
plt.scatter(
    x_plot,
    df_ERange['Electric Range'],
    alpha=0.7,
    s=100, # Size of the markers
    edgecolors='k', # Black outline
    c=make_indices, # Color points based on Make
    cmap='viridis' # Colormap to use for different makes
)

# Customization
plt.title('Electric Vehicle Range Distribution by Manufacturer', fontsize=16, fontweight='bold')
plt.xlabel('Manufacturer (Make)', fontsize=14)
plt.ylabel('EPA Electric Range (miles)', fontsize=14)
plt.xticks(ticks=np.arange(len(make_names)), labels=make_names, rotation=45, ha='right')
plt.axhline(110, color='r', linestyle='--', alpha=0.5, label='~110 Mile')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# 7. Brand-Specific EV Location Map

In [ ]:
df_recent = df_cleaned[(df_cleaned['Model Year']>=2015) & (df_cleaned['Model Year']<=2025)]
df_recent_popular = df_recent[df_recent['Make'].isin(popular_makes)]

In [ ]:
unique_categories = df_recent_popular['Make'].unique()

colors = plt.cm.get_cmap('tab10', len(unique_categories))
color_map = {make: colors(i) for i, make in enumerate(unique_categories)}


for make in unique_categories:
    # Filter the DataFrame for the current make
    df_filtered = df_recent_popular[df_recent_popular['Make'] == make].reset_index(drop=True)

    # Ensure Latitude and Longitude are numeric before calculating the mean.
    valid_latitudes = pd.to_numeric(df_filtered['Latitude'], errors='coerce').dropna()
    valid_longitudes = pd.to_numeric(df_filtered['Longitude'], errors='coerce').dropna()
    df_filtered.dropna(subset=['Latitude', 'Longitude'], inplace=True)

    # Calculate center point for better visual focus on the filtered data
    # Explicitly casting to float() ensures a scalar is passed.
    center_lat = float(valid_latitudes.mean())
    center_lon = float(valid_longitudes.mean())

     # --- Matplotlib Plotting ---

    fig, ax = plt.subplots(figsize=(15, 6))

    # Scatter plot: Longitude on X-axis, Latitude on Y-axis
    ax.scatter(
        df_filtered['Longitude'].values.astype(float),
        df_filtered['Latitude'].values.astype(float),
        label=make,
        color=color_map[make],
        s=100, # Marker size
        alpha=0.8
    )

    # Set title and labels
    ax.set_title(f"Location Plot for Brand: {make}", fontsize=14)
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
   # ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend()
    plt.show()


In [ ]:
df_locations = df_recent_popular.copy()
# Ensure Latitude and Longitude are numeric and clean the data once
df_locations.loc[:, 'Latitude'] = pd.to_numeric(df_locations['Latitude'], errors='coerce')
df_locations.loc[:, 'Longitude'] = pd.to_numeric(df_locations['Longitude'], errors='coerce')
df_locations.dropna(subset=['Latitude', 'Longitude'], inplace=True)

# Generate SINGLE Static Scatter Plot for All Makes

# Identify all unique brands/makes
unique_makes = df_locations['Make'].unique()

# Define a color map for visual consistency
colors = plt.cm.get_cmap('tab10', len(unique_makes))
color_map = {make: colors(i) for i, make in enumerate(unique_makes)}

# --- Initialize the single plot outside the loop ---
fig, ax = plt.subplots(figsize=(30, 10))

# --- Loop through each unique make and plot on the same axes ---
has_data = False

for make in unique_makes:
    # Filter the DataFrame for the current make
    # Use .reset_index(drop=True) to force a new DataFrame object and prevent the SettingWithCopyWarning.
    df_filtered = df_locations[df_locations['Make'] == make].reset_index(drop=True)

    # Check if we have valid data remaining
    if df_filtered.empty:
        continue

    # Scatter plot: Longitude on X-axis, Latitude on Y-axis
    ax.scatter(
        # Explicitly convert to numpy array of floats
        df_filtered['Longitude'].values.astype(float),
        df_filtered['Latitude'].values.astype(float),
        label=make, # Label is used for the legend
        color=color_map[make],
        s=100, # Marker size
        alpha=0.8
    )
    has_data = True

if has_data:
    # Set title and labels
    ax.set_title("Global Location Plot of All Brands", fontsize=16)
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)

    # Ensure aspect ratio is appropriate for coordinates (makes it look more like a map)
    ax.set_aspect('equal', adjustable='box')

    ax.grid(True, linestyle=':', alpha=0.5)

    # Add legend for all brands outside the plot area
    ax.legend(title="Car Make", loc='upper left', bbox_to_anchor=(1.0, 1.0))

    plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout to make room for the legend

    plt.show()
else:
    print("No valid data available to create the plots.")